# RL Experiment 03: Curriculum Learning

**Self-contained experiment notebook using DRY architecture.**

## Architecture (DRY Principle)

| Location | What | Example |
|----------|------|---------|
| `schedule_engine/notebooks/` | Reusable functions | `load_context()`, `create_env()`, `train_agent()` |
| `src/schedule_engine/rl/` | Production RL components | `ScheduleEnv`, PPO/DQN agents |
| **This notebook** | Experiment-specific config | Curriculum stages, execution |

## Experiment Overview
- **Agent**: PPO with curriculum learning
- **Goal**: Train in staged phases with increasing difficulty
- **Stages**: Easy → Medium → Hard (increasing episode length)
- **Metrics**: Per-stage convergence and final performance

## 1. Imports (from `schedule_engine/notebooks/`)

In [ ]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path

# DRY IMPORTS FROM schedule_engine/notebooks/
from schedule_engine.notebooks import (
    build_notebook_config,
    create_env,
    evaluate_agent,
    load_context,
    set_global_seed,
    train_agent,
)

print(" All imports from schedule_engine/notebooks/ successful!")

## 2. Configuration (Inline - Experiment-Specific)

In [ ]:
# ============================================================================
# RL EXPERIMENT 03 CONFIGURATION - Curriculum Learning Stages
# ============================================================================

SEED = 42
POP_SIZE = 20

# Curriculum stages: gradually increase difficulty
STAGES = [
    {"name": "easy", "max_generations": 30, "max_steps": 10, "timesteps": 3000},
    {"name": "medium", "max_generations": 50, "max_steps": 15, "timesteps": 4000},
    {"name": "hard", "max_generations": 80, "max_steps": 20, "timesteps": 5000},
]

# Paths - Organized by experiment with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/notebooks/rl_03_curriculum_{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Config: pop={POP_SIZE}, stages={len(STAGES)}")
print(f" Output: {OUTPUT_DIR}")
print(f" Stages: {[s['name'] for s in STAGES]}")

## 3. Load Data

In [ ]:
# Set reproducibility
set_global_seed(SEED)

# Build config and load scheduling context
config = build_notebook_config(seed=SEED, overrides={"pop_size": POP_SIZE})
_, context = load_context(DATA_DIR, config)

print(f" Scheduling context loaded")

## 4. Curriculum Training Loop

In [ ]:
# Train PPO with curriculum: reuse agent across stages
agent = None
stage_results = []
total_train_time = 0.0

for i, stage in enumerate(STAGES):
    print(f"\n{'='*50}")
    print(f"STAGE {i+1}/{len(STAGES)}: {stage['name'].upper()}")
    print(f"{'='*50}")
    
    # Create environment for this stage
    env = create_env(
        context=context,
        pop_size=POP_SIZE,
        max_generations=stage["max_generations"],
        max_steps=stage["max_steps"],
    )
    
    # Train or continue training
    if agent is None:
        agent, train_time = train_agent(
            agent_type="ppo",
            env=env,
            timesteps=stage["timesteps"],
            seed=SEED,
        )
    else:
        import time
        agent.set_env(env)
        start = time.time()
        agent.learn(total_timesteps=stage["timesteps"], progress_bar=False)
        train_time = time.time() - start
    
    total_train_time += train_time
    
    # Evaluate at this stage
    result = evaluate_agent(agent, env, max_generations=stage["max_generations"])
    stage_results.append({
        "stage": stage["name"],
        "train_time": train_time,
        "best_fitness": result.best_fitness,
        "convergence_gen": result.convergence_gen,
    })
    
    print(f" Stage {stage['name']}: best={result.best_fitness}, conv={result.convergence_gen} (train={train_time:.2f}s)")

## 5. Results Summary

In [ ]:
print(f"\n{'='*60}")
print(f"RL EXPERIMENT 03: CURRICULUM LEARNING RESULTS")
print(f"{'='*60}")
print(f"Total training time: {total_train_time:.2f}s")
print(f"\nPer-stage results:")
for sr in stage_results:
    print(f"  {sr['stage']:8s}: fitness={sr['best_fitness']}, conv={sr['convergence_gen']}, time={sr['train_time']:.2f}s")
print(f"{'='*60}")

## 6. Save Results

In [ ]:
import json

# Save experiment results
results_data = {
    "experiment": "rl_03_curriculum_learning",
    "timestamp": TIMESTAMP,
    "config": {
        "seed": SEED,
        "pop_size": POP_SIZE,
        "stages": STAGES,
    },
    "results": {
        "total_train_time_seconds": total_train_time,
        "stage_results": stage_results,
        "final_best_fitness": stage_results[-1]["best_fitness"],
        "final_convergence_gen": stage_results[-1]["convergence_gen"],
    },
}

results_path = OUTPUT_DIR / "results.json"
with open(results_path, "w") as f:
    json.dump(results_data, f, indent=2)

print(f" Results saved to: {results_path}")